# **Preprocesamiento datos de MiBici**

Empezamos cargando la información de los meses de enero a mayo de 2026 para el prototipo. Se combinan como un único dataframe, ya que en el origen están dividiso por mes.

In [ ]:
#Importamos librer+ias
import pandas as pd
import numpy as np

In [ ]:
# Listado de archivos presentes en el ambiente del notebook
csv_files = [
    "datos_abiertos_2026_01.csv",
    "datos_abiertos_2026_02.csv",
    "datos_abiertos_2026_03.csv",
    "datos_abiertos_2026_04.csv",
    "datos_abiertos_2026_05.csv",
]

dataframes = []

# Carga y normalización de cada archivo
for file_name in csv_files:
    try:
        df = pd.read_csv(file_name, encoding="utf-8")
    except UnicodeDecodeError:
        df = pd.read_csv(file_name, encoding="latin-1")

    # Corrección de codificación en encabezados
    df.columns = df.columns.str.replace(
        "AÃ±o_de_nacimiento", "Año_de_nacimiento"
    )

    dataframes.append(df)
    print(f"Cargado: {file_name} | Registros: {len(df):,}")

# Unificación en un solo DataFrame
df_mi_bici = pd.concat(dataframes, ignore_index=True)

# Conversión de tipos de datos de fecha
df_mi_bici["Inicio_del_viaje"] = pd.to_datetime(df_mi_bici["Inicio_del_viaje"])
df_mi_bici["Fin_del_viaje"] = pd.to_datetime(df_mi_bici["Fin_del_viaje"])

print(
    f"\nDataset unificado exitosamente: {df_mi_bici.shape[0]:,} filas x {df_mi_bici.shape[1]} columnas."
)

Cargado: datos_abiertos_2026_01.csv | Registros: 357,180
Cargado: datos_abiertos_2026_02.csv | Registros: 333,756
Cargado: datos_abiertos_2026_03.csv | Registros: 376,243
Cargado: datos_abiertos_2026_04.csv | Registros: 351,851
Cargado: datos_abiertos_2026_05.csv | Registros: 315,895

Dataset unificado exitosamente: 1,734,925 filas x 8 columnas.


Verificamos el nuevo dataframe y sus características

In [ ]:
df_mi_bici.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1734925 entries, 0 to 1734924
Data columns (total 8 columns):
 #   Column             Dtype         
---  ------             -----         
 0   Viaje_Id           int64         
 1   Usuario_Id         int64         
 2   Genero             object        
 3   Año_de_nacimiento  float64       
 4   Inicio_del_viaje   datetime64[ns]
 5   Fin_del_viaje      datetime64[ns]
 6   Origen_Id          int64         
 7   Destino_Id         int64         
dtypes: datetime64[ns](2), float64(1), int64(4), object(1)
memory usage: 105.9+ MB


In [ ]:
df_mi_bici.head()

,Viaje_Id,Usuario_Id,Genero,Año_de_nacimiento,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id
0,42349943,1726166,M,1967.0,2026-01-01 00:03:07,2026-01-01 00:03:39,64,64
1,42349944,1726166,M,1967.0,2026-01-01 00:04:05,2026-01-01 00:13:20,64,35
2,42349945,667524,M,1994.0,2026-01-01 00:05:24,2026-01-01 00:33:40,345,70
3,42349946,534776,M,1978.0,2026-01-01 00:05:48,2026-01-01 00:30:54,35,333
4,42349947,161552,M,1997.0,2026-01-01 00:07:07,2026-01-01 00:14:56,268,226


Para poder extrar información más útil (como la duración de cada viaje en minutos), procederemos a separar los timestamps de inicio y fin del viaje en fecha y hora. Esto nos separaá los datos en una fecha de inicio, una fecha de fin, una hora de inicio y una hora de fin. Después quitamos los timestamps originales para no meter ruido a los datos.

In [ ]:
df_mi_bici['Inicio_fecha'] = df_mi_bici['Inicio_del_viaje'].dt.date
df_mi_bici['Fin_fecha'] = df_mi_bici['Fin_del_viaje'].dt.date
df_mi_bici['Inicio_hora'] = df_mi_bici['Inicio_del_viaje'].dt.time
df_mi_bici['Fin_hora'] = df_mi_bici['Fin_del_viaje'].dt.time

# Eliminamos las columnas originales
df_mi_bici = df_mi_bici.drop(columns=['Inicio_del_viaje', 'Fin_del_viaje'])

# Definimos el nuevo orden colocando las nuevas columnas tras 'Año_de_nacimiento'
columnas = list(df_mi_bici.columns)
idx = columnas.index('Año_de_nacimiento') + 1

nuevas_cols = ['Inicio_fecha', 'Fin_fecha', 'Inicio_hora', 'Fin_hora']

# Removemos las nuevas columnas de sus posiciones anteriores para reinsertarlas
for col in nuevas_cols:
    columnas.remove(col)

# Insertamos las nuevas columnas agrupadas (fechas y luego horas) después de 'Año_de_nacimiento'
columnas[idx:idx] = nuevas_cols

# Aplicamos el nuevo orden al DataFrame
df_mi_bici = df_mi_bici[columnas]

# Mostramos el DataFrame con las modificaciones
df_mi_bici.head()

,Viaje_Id,Usuario_Id,Genero,Año_de_nacimiento,Inicio_fecha,Fin_fecha,Inicio_hora,Fin_hora,Origen_Id,Destino_Id
0,42349943,1726166,M,1967.0,2026-01-01,2026-01-01,00:03:07,00:03:39,64,64
1,42349944,1726166,M,1967.0,2026-01-01,2026-01-01,00:04:05,00:13:20,64,35
2,42349945,667524,M,1994.0,2026-01-01,2026-01-01,00:05:24,00:33:40,345,70
3,42349946,534776,M,1978.0,2026-01-01,2026-01-01,00:05:48,00:30:54,35,333
4,42349947,161552,M,1997.0,2026-01-01,2026-01-01,00:07:07,00:14:56,268,226


Verificamos nuevamente el dataframe

In [ ]:
df_mi_bici.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1734925 entries, 0 to 1734924
Data columns (total 10 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   Viaje_Id           1734925 non-null  int64  
 1   Usuario_Id         1734925 non-null  int64  
 2   Genero             1723532 non-null  object 
 3   Año_de_nacimiento  1734725 non-null  float64
 4   Inicio_fecha       1734925 non-null  object 
 5   Fin_fecha          1734925 non-null  object 
 6   Inicio_hora        1734925 non-null  object 
 7   Fin_hora           1734925 non-null  object 
 8   Origen_Id          1734925 non-null  int64  
 9   Destino_Id         1734925 non-null  int64  
dtypes: float64(1), int64(4), object(5)
memory usage: 132.4+ MB


Ahora agregaremos dos columnas útiles: minutos y edad. Minutos nos va a sevrir para medir más eficientemente la duración de un viaje. Edad nos va a servir para anonimizar un poco más los datos y proteger la identidad de los usuarios, así como agruparlos por un número más apto para procesamiento que el año de nacimiento.

In [ ]:
# 1. Calcular Duración_viaje_minutos
inicio_dt = pd.to_datetime(df_mi_bici['Inicio_fecha'].astype(str) + ' ' + df_mi_bici['Inicio_hora'].astype(str))
fin_dt = pd.to_datetime(df_mi_bici['Fin_fecha'].astype(str) + ' ' + df_mi_bici['Fin_hora'].astype(str))
df_mi_bici['Duración_viaje_minutos'] = ((fin_dt - inicio_dt).dt.total_seconds() / 60).round(2)

# 2. Calcular Edad basada en el año actual (2026) y la columna 'Año_de_nacimiento'
df_mi_bici['Edad'] = 2026 - df_mi_bici['Año_de_nacimiento']

# 3. Organizar las columnas en el orden solicitado
columnas = list(df_mi_bici.columns)

# Colocar 'Edad' a la derecha de 'Año_de_nacimiento' y luego borrar 'Año_de_nacimiento'
idx_nacimiento = columnas.index('Año_de_nacimiento')
columnas.insert(idx_nacimiento + 1, 'Edad')
columnas.remove('Año_de_nacimiento')

# Eliminar cualquier referencia duplicada de 'Edad' al final de la lista de columnas
while columnas.count('Edad') > 1:
    columnas.reverse()
    columnas.remove('Edad')
    columnas.reverse()

# Colocar 'Duración_viaje_minutos' a la derecha de 'Fin_hora'
columnas.remove('Duración_viaje_minutos')
idx_fin_hora = columnas.index('Fin_hora')
columnas.insert(idx_fin_hora + 1, 'Duración_viaje_minutos')

# Aplicar el orden final limpio al DataFrame
df_mi_bici = df_mi_bici.loc[:, ~df_mi_bici.columns.duplicated()]
df_mi_bici = df_mi_bici[columnas]

# Mostrar el DataFrame resultante
df_mi_bici.head()

,Viaje_Id,Usuario_Id,Genero,Edad,Inicio_fecha,Fin_fecha,Inicio_hora,Fin_hora,Duración_viaje_minutos,Origen_Id,Destino_Id
0,42349943,1726166,M,59.0,2026-01-01,2026-01-01,00:03:07,00:03:39,0.53,64,64
1,42349944,1726166,M,59.0,2026-01-01,2026-01-01,00:04:05,00:13:20,9.25,64,35
2,42349945,667524,M,32.0,2026-01-01,2026-01-01,00:05:24,00:33:40,28.27,345,70
3,42349946,534776,M,48.0,2026-01-01,2026-01-01,00:05:48,00:30:54,25.10,35,333
4,42349947,161552,M,29.0,2026-01-01,2026-01-01,00:07:07,00:14:56,7.82,268,226


In [ ]:
df_mi_bici.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1734925 entries, 0 to 1734924
Data columns (total 11 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   Viaje_Id                1734925 non-null  int64  
 1   Usuario_Id              1734925 non-null  int64  
 2   Genero                  1723532 non-null  object 
 3   Edad                    1734725 non-null  float64
 4   Inicio_fecha            1734925 non-null  object 
 5   Fin_fecha               1734925 non-null  object 
 6   Inicio_hora             1734925 non-null  object 
 7   Fin_hora                1734925 non-null  object 
 8   Duración_viaje_minutos  1734925 non-null  float64
 9   Origen_Id               1734925 non-null  int64  
 10  Destino_Id              1734925 non-null  int64  
dtypes: float64(2), int64(4), object(5)
memory usage: 145.6+ MB


# **Limpieza y análisis**

Ahora que ya tenemos datos en un formato mejor ajustado a nuestras necesidades, procedemos a explorarlos y hacer limpieza adicional.

Como primer punto buscaremos viajes atípicamente cortos. Estos pueden ser viajes con duración menor a un minuto y/o que estén en la misma estación de salida y llegada. El racional es que eto puede indicar que el usuario estaba por hacer el viaje y al final lo cancelo, o que abrió la palicaicón por error y estos nos puede meter ruido en el aprendizaje de nuestros modelos.

In [ ]:
# 1. Definir la condición de filtrado
condicion = (df_mi_bici["Duración_viaje_minutos"] < 1) & (
    df_mi_bici["Origen_Id"] == df_mi_bici["Destino_Id"]
)

# 2. Filtrar el DataFrame
viajes_mismo_origen_destino = df_mi_bici[condicion]

# 3. Mostrar el total de registros encontrados y las primeras filas
print(
    f"Se encontraron {len(viajes_mismo_origen_destino):,} viajes de menos de 1 minuto con el mismo origen y destino."
)
viajes_mismo_origen_destino.head()

Se encontraron 52,159 viajes de menos de 1 minuto con el mismo origen y destino.


,Viaje_Id,Usuario_Id,Genero,Edad,Inicio_fecha,Fin_fecha,Inicio_hora,Fin_hora,Duración_viaje_minutos,Origen_Id,Destino_Id
0,42349943,1726166,M,59.0,2026-01-01,2026-01-01,00:03:07,00:03:39,0.53,64,64
41,42349984,4630657,M,33.0,2026-01-01,2026-01-01,00:34:18,00:34:26,0.13,74,74
92,42350035,3501025,M,32.0,2026-01-01,2026-01-01,05:55:05,05:55:21,0.27,58,58
112,42350055,342106,M,52.0,2026-01-01,2026-01-01,06:13:03,06:13:51,0.80,163,163
117,42350060,1153265,F,44.0,2026-01-01,2026-01-01,06:20:18,06:20:52,0.57,323,323


Encontramos 52,159 viajes con duración menor a 1 minuto y con el mismo id de entrada y salida. Los borraremos bajo el racional previamente explicado.

In [ ]:
# 1. Definir la condición de filtrado (mismo origen y destino con duración menor a 1 minuto)
condicion = (df_mi_bici["Duración_viaje_minutos"] < 1) & (
    df_mi_bici["Origen_Id"] == df_mi_bici["Destino_Id"]
)

# 2. Borrar estas filas del DataFrame original manteniendo solo las que no cumplen esta condición
df_mi_bici = df_mi_bici[~condicion]

# 3. Mostrar el nuevo tamaño del DataFrame original
print(f"El DataFrame ahora tiene {df_mi_bici.shape[0]:,} filas y {df_mi_bici.shape[1]} columnas.")

El DataFrame ahora tiene 1,682,766 filas y 11 columnas.


Ahora separaremos las variables en numéricas y no numéricas, para poder hacer un análisis estadístico de nuestras variables y seguir explorando en búsqueda de datos atípicos

In [ ]:
# 1. Convertir a categórica/string cualquier columna que contenga "Id" o "ID" en su nombre
id_cols = [col for col in df_mi_bici.columns if "Id" in col or "ID" in col]
df_mi_bici[id_cols] = df_mi_bici[id_cols].astype(str)

print("Columnas con ID convertidas a categóricas:", id_cols)

# 2. Separación de las variables numéricas y de categoría para obtener estadísticas
bici_num = df_mi_bici.select_dtypes(include=np.number)
bici_cat = df_mi_bici.select_dtypes(exclude=np.number)

# 3. Mostrar las columnas en cada grupo
print("\nVariables numéricas:", list(bici_num.columns))
print("Variables no numéricas / categóricas:", list(bici_cat.columns))

Columnas con ID convertidas a categóricas: ['Viaje_Id', 'Usuario_Id', 'Origen_Id', 'Destino_Id']

Variables numéricas: ['Edad', 'Duración_viaje_minutos']
Variables no numéricas / categóricas: ['Viaje_Id', 'Usuario_Id', 'Genero', 'Inicio_fecha', 'Fin_fecha', 'Inicio_hora', 'Fin_hora', 'Origen_Id', 'Destino_Id']


In [ ]:
# Estadísticas descriptivas de variables numéricas
bici_data = bici_num.describe()
skewness_bici = bici_num.skew()
kurtosis_bici = bici_num.kurt()

skewness_df = pd.DataFrame({"skew": skewness_bici}).T
kurtosis_df = pd.DataFrame({"kurt": kurtosis_bici}).T

# Agregando asimetría (skew) y curtosis (kurt) a las estadísticas descriptivas
bici_kurt_skew = pd.concat([bici_data, kurtosis_df, skewness_df])

# Transponer para visualizar cada variable en una fila
bici_kurt_skew.T

,count,mean,std,min,25%,50%,75%,max,kurt,skew
Edad,1682580.0,35.579994,19.516153,16.0,27.00,33.00,42.00,1827.00,5609.034840,61.315758
Duración_viaje_minutos,1682766.0,11.500411,10.105629,0.0,6.05,9.82,15.32,5920.05,73028.580461,144.704682


1. Encontramos que la edad máxima encontrada en nuestros datos es de 1827, lo cual es imposible. Esto pude deberse a un error en la captura del año de nacimiento dle usuario, por lo que al hacer el cálculo de la edad, el número fue demasiado grande.

2. Con los minutos de duración tenemos casos similares. Hay reportes de viajes que duraron más de 4 días y unos que no tienen duración (a pesar de la limpeiza previa que hicimos). Esto puede deberse a que el usuario olvidó terminar el viaje en la aplicación o, en algo más improibable, genuinamente tuvo la bicleta todo ese tiempo.

Procederemos a buscar esos casos atípicos.

In [ ]:
# Total de filas en el dataset actual
total_filas = len(df_mi_bici)

# 1. Definir los filtros/condiciones
cond_edad_alta = df_mi_bici["Edad"] > 100
cond_duracion_cero = df_mi_bici["Duración_viaje_minutos"] < 1
cond_duracion_alta = df_mi_bici["Duración_viaje_minutos"] > 240  # Más de 4 horas

# 2. Conteo de registros por cada condición
cant_edad_alta = cond_edad_alta.sum()
cant_duracion_cero = cond_duracion_cero.sum()
cant_duracion_alta = cond_duracion_alta.sum()

# Conteo de registros con al menos una anomalía (combinados)
cond_cualquiera = cond_edad_alta | cond_duracion_cero | cond_duracion_alta
cant_total_anomalias = cond_cualquiera.sum()

# 3. Imprimir el resumen diagnóstico
print(f"--- DIAGNÓSTICO DE VALORES ATÍPICOS ---")
print(f"Total de registros analizados: {total_filas:,}\n")

print(
    f"1. Edad > 100 años: {cant_edad_alta:,} registros ({cant_edad_alta / total_filas:.2%})"
)
print(
    f"2. Duración < 0 minutos: {cant_duracion_cero:,} registros ({cant_duracion_cero / total_filas:.2%})"
)
print(
    f"3. Duración > 240 minutos (4h): {cant_duracion_alta:,} registros ({cant_duracion_alta / total_filas:.2%})"
)

print(f"\n" + "-" * 40)
print(
    f"TOTAL ÚNICO CON ANOMALÍAS: {cant_total_anomalias:,} registros ({cant_total_anomalias / total_filas:.2%})"
)

--- DIAGNÓSTICO DE VALORES ATÍPICOS ---
Total de registros analizados: 1,682,766

1. Edad > 100 años: 133 registros (0.01%)
2. Duración < 0 minutos: 5,411 registros (0.32%)
3. Duración > 240 minutos (4h): 108 registros (0.01%)

----------------------------------------
TOTAL ÚNICO CON ANOMALÍAS: 5,650 registros (0.34%)


Emcontramos qjue las anomlaías representan menos del 1% del total de los datos del dataset, por lo que podemos borrarlos con un impacto menor al comportamiento del modelo.

In [ ]:
# 1. Definir la condición que agrupa las anomalías a eliminar
cond_anomalias = (
    (df_mi_bici["Edad"] > 100)
    | (df_mi_bici["Duración_viaje_minutos"] < 1)
    | (df_mi_bici["Duración_viaje_minutos"] > 240)
)

# 2. Guardar conteo previo y filtrar el DataFrame conservando los registros válidos (~cond_anomalias)
filas_antes = len(df_mi_bici)
df_mi_bici = df_mi_bici[~cond_anomalias].reset_index(drop=True)
filas_despues = len(df_mi_bici)

# 3. Confirmación de eliminación
print(f"Registros atípicos eliminados: {filas_antes - filas_despues:,}")
print(f"Total de registros restantes en dataset limpio: {filas_despues:,}\n")

# 4. Mostrar la información del DataFrame limpio
df_mi_bici.info(show_counts=True)

Registros atípicos eliminados: 5,650
Total de registros restantes en dataset limpio: 1,677,116

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1677116 entries, 0 to 1677115
Data columns (total 11 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   Viaje_Id                1677116 non-null  object 
 1   Usuario_Id              1677116 non-null  object 
 2   Genero                  1666027 non-null  object 
 3   Edad                    1676931 non-null  float64
 4   Inicio_fecha            1677116 non-null  object 
 5   Fin_fecha               1677116 non-null  object 
 6   Inicio_hora             1677116 non-null  object 
 7   Fin_hora                1677116 non-null  object 
 8   Duración_viaje_minutos  1677116 non-null  float64
 9   Origen_Id               1677116 non-null  object 
 10  Destino_Id              1677116 non-null  object 
dtypes: float64(2), object(9)
memory usage: 140.7+ MB


Revisamos nuevamente estadísticas de los datos como última comprobación

In [ ]:
# 1. Volver a seleccionar las variables numéricas del DataFrame limpio actual
bici_num_limpio = df_mi_bici.select_dtypes(include=np.number)

# 2. Calcular las estadísticas descriptivas, asimetría y curtosis tras la limpieza
bici_data_limpia = bici_num_limpio.describe()
skewness_limpia = bici_num_limpio.skew()
kurtosis_limpia = bici_num_limpio.kurt()

skewness_df = pd.DataFrame({"skew": skewness_limpia}).T
kurtosis_df = pd.DataFrame({"kurt": kurtosis_limpia}).T

# 3. Consolidar el resumen y transponerlo para su visualización
bici_stats_nuevas = pd.concat(
    [bici_data_limpia, kurtosis_df, skewness_df]
)
bici_stats_nuevas.T

,count,mean,std,min,25%,50%,75%,max,kurt,skew
Edad,1676931.0,35.439943,11.278784,16.0,27.00,33.00,42.00,87.00,0.507060,0.976956
Duración_viaje_minutos,1677116.0,11.500727,7.542426,1.0,6.08,9.83,15.33,238.57,37.114579,2.910379


In [ ]:
df_mi_bici.head()

,Viaje_Id,Usuario_Id,Genero,Edad,Inicio_fecha,Fin_fecha,Inicio_hora,Fin_hora,Duración_viaje_minutos,Origen_Id,Destino_Id
0,42349944,1726166,M,59.0,2026-01-01,2026-01-01,00:04:05,00:13:20,9.25,64,35
1,42349945,667524,M,32.0,2026-01-01,2026-01-01,00:05:24,00:33:40,28.27,345,70
2,42349946,534776,M,48.0,2026-01-01,2026-01-01,00:05:48,00:30:54,25.10,35,333
3,42349947,161552,M,29.0,2026-01-01,2026-01-01,00:07:07,00:14:56,7.82,268,226
4,42349948,330724,M,33.0,2026-01-01,2026-01-01,00:07:05,00:30:35,23.50,54,32


Ahora nuestro datos están listo para continuar hacia la creación del modelo

### Construcción de Variables para P1: Predicción de Demanda
Agrupamos la cantidad de viajes salientes por estación de origen (`Origen_Id`) y por hora.

In [ ]:
# Extraer la hora a partir de 'Inicio_hora' (que es de tipo datetime.time o string) para poder agrupar por hora
# Como 'Inicio_hora' ya existe en tu df_mi_bici, obtenemos el componente de la hora (hour)
df_mi_bici['Hora_Inicio'] = pd.to_datetime(df_mi_bici['Inicio_hora'].astype(str)).dt.hour

# La columna 'Inicio_fecha' ya contiene la fecha y 'Edad' ya está calculada en df_mi_bici
# Agrupar viajes salientes (Demanda) utilizando las columnas reales de tu DataFrame
p1_demanda = df_mi_bici.groupby(['Origen_Id', 'Inicio_fecha', 'Hora_Inicio', 'Edad']).size().reset_index(name='Cantidad_Viajes_Salientes')

print("Ejemplo de variables para P1 (Demanda) adaptado:")
display(p1_demanda.head())

/tmp/ipykernel_2044/1705429050.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_mi_bici['Hora_Inicio'] = pd.to_datetime(df_mi_bici['Inicio_hora'].astype(str)).dt.hour


Ejemplo de variables para P1 (Demanda) adaptado:


,Origen_Id,Inicio_fecha,Hora_Inicio,Edad,Cantidad_Viajes_Salientes
0,10,2026-01-01,0,37.0,1
1,10,2026-01-01,7,29.0,1
2,10,2026-01-01,10,37.0,1
3,10,2026-01-01,14,52.0,1
4,10,2026-01-01,15,29.0,1


In [ ]:
p1_demanda.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1546546 entries, 0 to 1546545
Data columns (total 5 columns):
 #   Column                     Non-Null Count    Dtype  
---  ------                     --------------    -----  
 0   Origen_Id                  1546546 non-null  object 
 1   Inicio_fecha               1546546 non-null  object 
 2   Hora_Inicio                1546546 non-null  int32  
 3   Edad                       1546546 non-null  float64
 4   Cantidad_Viajes_Salientes  1546546 non-null  int64  
dtypes: float64(1), int32(1), int64(1), object(2)
memory usage: 53.1+ MB


### Construcción de Variables para P2: Saturación / Presión Operativa (Proxy)
Calculamos el balance neto (Llegadas - Salidas) acumulado por estación y hora para medir la presión en la operación.

In [ ]:
# Extraer la hora numérica a partir de las columnas de hora ya existentes en df_mi_bici
df_mi_bici['Hora_Inicio_Num'] = pd.to_datetime(df_mi_bici['Inicio_hora'].astype(str)).dt.hour
df_mi_bici['Hora_Fin_Num'] = pd.to_datetime(df_mi_bici['Fin_hora'].astype(str)).dt.hour

# Contar salidas (salidas por origen, fecha y hora de inicio)
salidas = df_mi_bici.groupby(['Origen_Id', 'Inicio_fecha', 'Hora_Inicio_Num']).size().reset_index(name='Salidas')
salidas.columns = ['Estacion_Id', 'Fecha', 'Hora', 'Salidas']

# Contar entradas (llegadas por destino, fecha y hora de fin)
entradas = df_mi_bici.groupby(['Destino_Id', 'Fin_fecha', 'Hora_Fin_Num']).size().reset_index(name='Entradas')
entradas.columns = ['Estacion_Id', 'Fecha', 'Hora', 'Entradas']

# Unir flujos de entradas y salidas
flujo_estaciones = pd.merge(entradas, salidas, on=['Estacion_Id', 'Fecha', 'Hora'], how='outer').fillna(0)

# Calcular balance neto (Entradas - Salidas)
flujo_estaciones['Balance_Neto'] = flujo_estaciones['Entradas'] - flujo_estaciones['Salidas']

# Limpiar columnas auxiliares agregadas al dataframe original
df_mi_bici = df_mi_bici.drop(columns=['Hora_Inicio_Num', 'Hora_Fin_Num'], errors='ignore')

print("Ejemplo de variables para P2 (Balance Neto / Presión Operativa) adaptado:")
display(flujo_estaciones.head())

/tmp/ipykernel_2044/3268718061.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_mi_bici['Hora_Inicio_Num'] = pd.to_datetime(df_mi_bici['Inicio_hora'].astype(str)).dt.hour
/tmp/ipykernel_2044/3268718061.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_mi_bici['Hora_Fin_Num'] = pd.to_datetime(df_mi_bici['Fin_hora'].astype(str)).dt.hour


Ejemplo de variables para P2 (Balance Neto / Presión Operativa) adaptado:


,Estacion_Id,Fecha,Hora,Entradas,Salidas,Balance_Neto
0,10,2026-01-01,0,0.0,1.0,-1.0
1,10,2026-01-01,7,0.0,1.0,-1.0
2,10,2026-01-01,10,0.0,1.0,-1.0
3,10,2026-01-01,11,1.0,0.0,1.0
4,10,2026-01-01,14,0.0,1.0,-1.0


### Construcción de la Variable Binaria para P2: Riesgo de Capacidad (t+1)
Creamos un proxy de estado crítico basado en extremos del balance acumulado diario por estación y lo desplazamos una hora en el futuro para definir nuestra variable a predecir.

In [ ]:
# 1. Ordenar datos cronológicamente por estación
flujo_estaciones = flujo_estaciones.sort_values(by=['Estacion_Id', 'Fecha', 'Hora']).reset_index(drop=True)

# 2. Calcular el balance acumulado en el día para estimar el llenado/vaciado relativo
flujo_estaciones['Balance_Acumulado_Dia'] = flujo_estaciones.groupby(['Estacion_Id', 'Fecha'])['Balance_Neto'].cumsum()

# 3. Definir límites proxy de riesgo basados en los percentiles de la propia estación
# (Estaciones con balances acumulados muy negativos están en riesgo de desabasto; muy positivos en saturación)
limites = flujo_estaciones.groupby('Estacion_Id')['Balance_Acumulado_Dia'].agg([lambda x: x.quantile(0.1), lambda x: x.quantile(0.9)]).reset_index()
limites.columns = ['Estacion_Id', 'Umbral_Desabasto', 'Umbral_Saturacion']

flujo_con_umbrales = pd.merge(flujo_estaciones, limites, on='Estacion_Id', how='left')

# 4. Crear la variable de estado actual (1: Riesgo por desabasto/saturación, 0: Normal)
flujo_con_umbrales['Estado_Critico'] = ((
    (flujo_con_umbrales['Balance_Acumulado_Dia'] <= flujo_con_umbrales['Umbral_Desabasto']) |
    (flujo_con_umbrales['Balance_Acumulado_Dia'] >= flujo_con_umbrales['Umbral_Saturacion'])
) & (flujo_con_umbrales['Balance_Neto'] != 0)).astype(int)

# 5. Crear la variable objetivo (Target) para la Próxima Hora (t+1)
# Desplazamos (shift) el estado crítico una hora hacia atrás por estación para que represente el riesgo en t+1
flujo_con_umbrales['Target_Riesgo_Proxima_Hora'] = flujo_con_umbrales.groupby(['Estacion_Id', 'Fecha'])['Estado_Critico'].shift(-1)

# Eliminar filas con NaN en el target debido al desplazamiento al final del día
p2_clasificacion = flujo_con_umbrales.dropna(subset=['Target_Riesgo_Proxima_Hora']).copy()
p2_clasificacion['Target_Riesgo_Proxima_Hora'] = p2_clasificacion['Target_Riesgo_Proxima_Hora'].astype(int)

print("Distribución de la variable objetivo (0: Normal, 1: Riesgo en t+1):")
print(p2_clasificacion['Target_Riesgo_Proxima_Hora'].value_counts(normalize=True) * 100)

print("\nEjemplo de variables preparadas para P2:")
display(p2_clasificacion[['Estacion_Id', 'Fecha', 'Hora', 'Balance_Neto', 'Balance_Acumulado_Dia', 'Estado_Critico', 'Target_Riesgo_Proxima_Hora']].head(10))

Distribución de la variable objetivo (0: Normal, 1: Riesgo en t+1):
Target_Riesgo_Proxima_Hora
0    78.105728
1    21.894272
Name: proportion, dtype: float64

Ejemplo de variables preparadas para P2:


,Estacion_Id,Fecha,Hora,Balance_Neto,Balance_Acumulado_Dia,Estado_Critico,Target_Riesgo_Proxima_Hora
0,10,2026-01-01,0,-1.0,-1.0,0,0
1,10,2026-01-01,7,-1.0,-2.0,0,0
2,10,2026-01-01,10,-1.0,-3.0,0,0
3,10,2026-01-01,11,1.0,-2.0,0,0
4,10,2026-01-01,14,-1.0,-3.0,0,0
5,10,2026-01-01,15,-1.0,-4.0,0,0
7,10,2026-01-02,6,-1.0,-1.0,0,0
8,10,2026-01-02,7,0.0,-1.0,0,0
9,10,2026-01-02,8,-2.0,-3.0,0,0
10,10,2026-01-02,9,-2.0,-5.0,0,0


### Construcción de Variables para P3: Agrupamiento (Clustering) de Estaciones
Determinamos el perfil de uso de cada estación calculando el porcentaje promedio de viajes que ocurren en cada una de las 24 horas del día.

In [ ]:
# Extraer de forma temporal las horas numéricas para el agrupamiento
horas_inicio_temp = pd.to_datetime(df_mi_bici['Inicio_hora'].astype(str)).dt.hour

# Crear una tabla cruzada de Estación de Origen vs Hora de Inicio usando las variables correctas
perfil_horas = pd.crosstab(df_mi_bici['Origen_Id'], horas_inicio_temp)

# Convertir las frecuencias absolutas a proporciones (porcentajes) por fila
perfil_horas_porcentaje = perfil_horas.div(perfil_horas.sum(axis=1), axis=0) * 100

# Renombrar columnas para claridad
perfil_horas_porcentaje.columns = [f'Pct_Viajes_Hora_{h}' for h in perfil_horas_porcentaje.columns]

print("Perfil horario de las estaciones para P3 (Clustering) adaptado:")
display(perfil_horas_porcentaje.head())

/tmp/ipykernel_2044/3340784254.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  horas_inicio_temp = pd.to_datetime(df_mi_bici['Inicio_hora'].astype(str)).dt.hour


Perfil horario de las estaciones para P3 (Clustering) adaptado:


,Pct_Viajes_Hora_0,Pct_Viajes_Hora_1,Pct_Viajes_Hora_5,Pct_Viajes_Hora_6,Pct_Viajes_Hora_7,Pct_Viajes_Hora_8,Pct_Viajes_Hora_9,Pct_Viajes_Hora_10,Pct_Viajes_Hora_11,Pct_Viajes_Hora_12,...,Pct_Viajes_Hora_14,Pct_Viajes_Hora_15,Pct_Viajes_Hora_16,Pct_Viajes_Hora_17,Pct_Viajes_Hora_18,Pct_Viajes_Hora_19,Pct_Viajes_Hora_20,Pct_Viajes_Hora_21,Pct_Viajes_Hora_22,Pct_Viajes_Hora_23
Origen_Id,,,,,,,,,,,,,,,,,,,,,
10,0.283072,0.0,0.490659,5.623703,8.416682,8.926212,5.529345,4.529156,4.359313,6.265333,...,6.340819,5.189658,4.642385,8.246839,7.303265,6.435176,5.340630,3.151538,1.490847,0.509530
100,0.085616,0.0,0.256849,10.616438,5.565068,8.647260,10.273973,2.996575,4.537671,7.448630,...,5.650685,7.020548,6.849315,9.589041,5.308219,3.938356,3.510274,2.140411,0.599315,0.256849
101,0.418994,0.0,3.910615,4.469274,2.932961,8.798883,10.474860,10.754190,6.983240,3.770950,...,4.050279,6.983240,5.865922,7.262570,6.983240,5.167598,3.351955,1.955307,1.536313,1.536313
102,0.042790,0.0,0.534874,2.439024,4.899444,9.520753,6.118956,5.648267,5.798032,5.498502,...,6.910569,7.787762,5.733847,6.803594,6.225931,6.546855,5.091998,4.557125,2.417629,0.984168
103,0.549753,0.0,0.054975,0.494777,3.573392,10.335349,13.304013,5.552501,4.892798,4.013194,...,4.288070,12.424409,10.995052,9.070918,4.837823,2.528862,3.518417,2.253986,1.869159,1.319406
